In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

# Si no los tienes instalados:
# pip install xgboost lightgbm
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

eps = 1e-6

# =========================
# 1. Load training datasets
# =========================
water_quality = pd.read_csv("../../data/water_quality_training_dataset.csv")
landsat = pd.read_csv("../../data/landsat_features_training.csv")
terraclimate = pd.read_csv("../../data/terraclimate_features_training.csv")

# =========================
# 2. Convert dates
# =========================
water_quality["Sample Date"] = pd.to_datetime(water_quality["Sample Date"], dayfirst=True)
landsat["Sample Date"] = pd.to_datetime(landsat["Sample Date"], dayfirst=True)
terraclimate["Sample Date"] = pd.to_datetime(terraclimate["Sample Date"], dayfirst=True)

# =========================
# 3. Create temporal features
# =========================
water_quality["month"] = water_quality["Sample Date"].dt.month
water_quality["year"] = water_quality["Sample Date"].dt.year
water_quality["dayofyear"] = water_quality["Sample Date"].dt.dayofyear

# =========================
# 4. Merge training datasets
# =========================
df = water_quality.merge(
    landsat,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df = df.merge(
    terraclimate,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 5. v4.0 Feature engineering ONLY
# =========================
df["nir_swir16_ratio"] = df["nir"] / (df["swir16"] + eps)
df["nir_swir22_ratio"] = df["nir"] / (df["swir22"] + eps)
df["green_nir_ratio"] = df["green"] / (df["nir"] + eps)

df["nir_minus_swir16"] = df["nir"] - df["swir16"]
df["nir_minus_green"] = df["nir"] - df["green"]

df["ndmi_pet"] = df["NDMI"] * df["pet"]
df["mndwi_pet"] = df["MNDWI"] * df["pet"]

df["swir_ratio"] = df["swir16"] / (df["swir22"] + eps)

df["nir_pet"] = df["nir"] * df["pet"]
df["swir16_pet"] = df["swir16"] * df["pet"]
df["ndmi_day"] = df["NDMI"] * df["dayofyear"]

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# =========================
# 6. Handle missing values
# =========================
train_medians = df.median(numeric_only=True)
df.fillna(train_medians, inplace=True)

# =========================
# 7. Define training features and targets
# =========================
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

feature_drop = targets + ["Sample Date", "Latitude", "Longitude"]
X = df.drop(columns=feature_drop)
y = df[targets].copy()

X_train_medians = X.median(numeric_only=True)

print("Training shape:", X.shape)
print("Targets:", targets)

# =========================
# 8. Load submission datasets
# =========================
submission = pd.read_csv("../../data/submission_template.csv")
landsat_val = pd.read_csv("../../data/landsat_features_validation.csv")
terraclimate_val = pd.read_csv("../../data/terraclimate_features_validation.csv")

# =========================
# 9. Convert dates
# =========================
submission["Sample Date"] = pd.to_datetime(submission["Sample Date"], dayfirst=True)
landsat_val["Sample Date"] = pd.to_datetime(landsat_val["Sample Date"], dayfirst=True)
terraclimate_val["Sample Date"] = pd.to_datetime(terraclimate_val["Sample Date"], dayfirst=True)

# =========================
# 10. Create temporal features
# =========================
submission["month"] = submission["Sample Date"].dt.month
submission["year"] = submission["Sample Date"].dt.year
submission["dayofyear"] = submission["Sample Date"].dt.dayofyear

# =========================
# 11. Merge validation datasets
# =========================
df_val = submission.merge(
    landsat_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df_val = df_val.merge(
    terraclimate_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 12. Same v4.0 feature engineering
# =========================
df_val["nir_swir16_ratio"] = df_val["nir"] / (df_val["swir16"] + eps)
df_val["nir_swir22_ratio"] = df_val["nir"] / (df_val["swir22"] + eps)
df_val["green_nir_ratio"] = df_val["green"] / (df_val["nir"] + eps)

df_val["nir_minus_swir16"] = df_val["nir"] - df_val["swir16"]
df_val["nir_minus_green"] = df_val["nir"] - df_val["green"]

df_val["ndmi_pet"] = df_val["NDMI"] * df_val["pet"]
df_val["mndwi_pet"] = df_val["MNDWI"] * df_val["pet"]

df_val["swir_ratio"] = df_val["swir16"] / (df_val["swir22"] + eps)

df_val["nir_pet"] = df_val["nir"] * df_val["pet"]
df_val["swir16_pet"] = df_val["swir16"] * df_val["pet"]
df_val["ndmi_day"] = df_val["NDMI"] * df_val["dayofyear"]

df_val.replace([np.inf, -np.inf], np.nan, inplace=True)

# =========================
# 13. Handle missing values
# =========================
df_val.fillna(train_medians, inplace=True)

# =========================
# 14. Prepare validation features
# =========================
X_val = df_val.drop(
    columns=[
        "Sample Date",
        "Latitude",
        "Longitude",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ],
    errors="ignore"
)

X_val = X_val.reindex(columns=X.columns)
X_val = X_val.fillna(X_train_medians)

print("Validation shape:", X_val.shape)

# =========================
# 15. Model definitions
# =========================
rf_params = dict(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

xgb_params = dict(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

lgb_params = dict(
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

# =========================
# 16. Train one model per target
# =========================
rf_preds = np.zeros((len(X_val), len(targets)))
xgb_preds = np.zeros((len(X_val), len(targets)))
lgb_preds = np.zeros((len(X_val), len(targets)))

rf_models = {}
xgb_models = {}
lgb_models = {}

for i, target in enumerate(targets):
    print(f"\nTraining target: {target}")

    y_target = y[target]

    # RandomForest
    rf_model = RandomForestRegressor(**rf_params)
    rf_model.fit(X, y_target)
    rf_preds[:, i] = rf_model.predict(X_val)
    rf_models[target] = rf_model

    # XGBoost
    xgb_model = XGBRegressor(**xgb_params)
    xgb_model.fit(X, y_target)
    xgb_preds[:, i] = xgb_model.predict(X_val)
    xgb_models[target] = xgb_model

    # LightGBM
    lgb_model = LGBMRegressor(**lgb_params)
    lgb_model.fit(X, y_target)
    lgb_preds[:, i] = lgb_model.predict(X_val)
    lgb_models[target] = lgb_model

print("\nAll target-specific models trained.")

# =========================
# 17. Ensembles
# =========================
# Simple average
ensemble_simple_preds = (rf_preds + xgb_preds + lgb_preds) / 3.0

# Weighted average
# Base weights: RF still gets highest weight because v4.0 was your best proven model
w_rf = 0.50
w_xgb = 0.25
w_lgb = 0.25

ensemble_weighted_preds = (
    w_rf * rf_preds +
    w_xgb * xgb_preds +
    w_lgb * lgb_preds
)

# =========================
# 18. Helper to build submissions
# =========================
def build_submission(base_submission, preds, out_path):
    sub = base_submission.copy()
    sub["Total Alkalinity"] = preds[:, 0]
    sub["Electrical Conductance"] = preds[:, 1]
    sub["Dissolved Reactive Phosphorus"] = preds[:, 2]

    sub = sub[
        [
            "Longitude",
            "Latitude",
            "Sample Date",
            "Total Alkalinity",
            "Electrical Conductance",
            "Dissolved Reactive Phosphorus"
        ]
    ]

    sub.to_csv(out_path, index=False)
    return sub

# =========================
# 19. Export submissions
# =========================
submission_rf = build_submission(
    submission,
    rf_preds,
    "../../submissions/submission_rf_targets.csv"
)

submission_xgb = build_submission(
    submission,
    xgb_preds,
    "../../submissions/submission_xgb_targets.csv"
)

submission_lgb = build_submission(
    submission,
    lgb_preds,
    "../../submissions/submission_lgb_targets.csv"
)

submission_ensemble_simple = build_submission(
    submission,
    ensemble_simple_preds,
    "../../submissions/submission_ensemble_simple.csv"
)

submission_ensemble_weighted = build_submission(
    submission,
    ensemble_weighted_preds,
    "../../submissions/submission_ensemble_weighted.csv"
)

# =========================
# 20. Quick checks
# =========================
print("\nRF submission preview:")
print(submission_rf.head())

print("\nXGB submission preview:")
print(submission_xgb.head())

print("\nLGB submission preview:")
print(submission_lgb.head())

print("\nEnsemble simple preview:")
print(submission_ensemble_simple.head())

print("\nEnsemble weighted preview:")
print(submission_ensemble_weighted.head())

print("\nMissing values check:")
print(submission_ensemble_weighted.isna().sum())

Training shape: (9319, 21)
Targets: ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
Validation shape: (200, 21)

Training target: Total Alkalinity
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000416 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4859
[LightGBM] [Info] Number of data points in the train set: 9319, number of used features: 21
[LightGBM] [Info] Start training from score 119.108208

Training target: Electrical Conductance
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000450 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4859
[LightGBM] [Info] Number of data points in the train set: 9319, number of used features: 21
[LightGBM] [Info] Start training from score 485.004146

Training target: Dissolved Re